# BookCorpus Analysis & Cleaning

Analyses `rojagtap/bookcorpus` and decides on filtering rules before combining with `thekingslee/9ja-bookcorpus`.

## Approach
Reuses the same 3-layer pipeline from `01_Refine_Scraped_Data.ipynb` (which produced the 9ja corpus), then adds BookCorpus-specific filters:

| Layer | From PDF scraper? | What it catches |
|-------|------------------|-----------------|
| 1 - Sanity (alpha ratio + word length) | Reused | OCR noise, gibberish |
| 2 - Language check (lingua) | Reused | Non-English fragments |
| 3 - Final ratio + min char length | Reused | Short or garbled chunks |
| 4 - Min word count | NEW (BookCorpus-specific) | Single / 1-4 word fragments |
| 5 - Boilerplate patterns | NEW (BookCorpus-specific) | Chapter headers, ISBNs, dividers |
| 6 - High punctuation filter | NEW (BookCorpus-specific) | Dialogue fragments, ellipsis lines |
| 7 - Exact deduplication | NEW (BookCorpus-specific) | Re-uploaded book duplicates |

> **Why these extra layers?**
> BookCorpus is sentence-split, producing many 1-4 word fragments and bare structural markers
> (Chapter One, PART II) not present in the paragraph-level PDF scrape.

## 0. Setup

In [ ]:
import re
import random
import hashlib
from functools import lru_cache

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

pd.set_option('display.max_colwidth', 300)
SAMPLE_ROWS = 5_000
print('Setup complete.')

## 1. Load Sample via Streaming

Uses `streaming=True` so only 5,000 rows are pulled without downloading the full 4.6 GB shard first.
See `03_FineWeb_Edu_Exploration.ipynb` for why streaming is necessary here.

In [ ]:
stream = load_dataset('rojagtap/bookcorpus', split='train', streaming=True)
rows = list(stream.take(SAMPLE_ROWS))
df_raw = pd.DataFrame(rows)

print(f'Columns : {list(df_raw.columns)}')
print(f'Rows    : {len(df_raw)}')
df_raw.head(10)

## 2. Raw Data Profile

Baseline stats before any filtering.

In [ ]:
texts_raw = df_raw['text'].tolist()

df_raw['char_len']   = df_raw['text'].str.len()
df_raw['word_count'] = df_raw['text'].str.split().str.len()
df_raw['alpha_ratio'] = df_raw['text'].apply(
    lambda t: sum(c.isalpha() for c in t) / max(len(t), 1)
)

print(df_raw[['char_len', 'word_count', 'alpha_ratio']].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_raw['char_len'].clip(upper=300).plot(kind='hist', bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Character length (clipped @ 300)')
df_raw['word_count'].clip(upper=60).plot(kind='hist', bins=40, ax=axes[1], color='coral')
axes[1].set_title('Word count (clipped @ 60)')
df_raw['alpha_ratio'].plot(kind='hist', bins=40, ax=axes[2], color='seagreen')
axes[2].set_title('Alphabetic character ratio')
plt.tight_layout()
plt.show()

In [ ]:
short_pct = (df_raw['word_count'] < 5).mean() * 100
print(f'Rows with < 5 words: {short_pct:.1f}%')
print('\n--- 20 shortest rows ---')
print(df_raw.nsmallest(20, 'word_count')[['text', 'word_count']].to_string(index=False))

## 3. Layer 1 - Sanity Filter (reused from `01_Refine_Scraped_Data.ipynb`)

**Rule**: >= 70% alphabetic characters AND average word length between 2-12 chars.

In [ ]:
def is_mostly_text(chunk, threshold=0.7):
    letters = sum(c.isalpha() for c in chunk)
    return letters / max(len(chunk), 1) >= threshold

def has_reasonable_word_lengths(chunk, min_avg=2, max_avg=12):
    words = chunk.split()
    if not words:
        return False
    avg_len = sum(len(w) for w in words) / len(words)
    return min_avg <= avg_len <= max_avg

def passes_sanity(chunk):
    return is_mostly_text(chunk) and has_reasonable_word_lengths(chunk)

after_l1 = [t for t in texts_raw if passes_sanity(t)]
print(f'Layer 1 -- kept {len(after_l1):,} / {len(texts_raw):,}  ({len(after_l1)/len(texts_raw)*100:.1f}%)')

## 4. Layer 2 - Language Check (reused from `01_Refine_Scraped_Data.ipynb`)

**Rule**: lingua detects English. Chunks < 20 chars skip detection (kept if they passed Layer 1).

In [ ]:
from lingua import Language, LanguageDetectorBuilder

@lru_cache(maxsize=1)
def _lang_detector():
    return LanguageDetectorBuilder.from_all_spoken_languages().build()

def passes_english_check(text, min_chars=20):
    t = text.strip()
    if len(t) < min_chars:
        return True
    lang = _lang_detector().detect_language_of(t)
    return lang == Language.ENGLISH

after_l2 = [t for t in after_l1 if passes_english_check(t)]
print(f'Layer 2 -- kept {len(after_l2):,} / {len(after_l1):,}  ({len(after_l2)/len(after_l1)*100:.1f}%)')

## 5. Layer 3 - Final Ratio + Min Length (reused from `01_Refine_Scraped_Data.ipynb`)

**Rule**: alphabetic ratio >= 0.6 AND length >= 10 characters.

In [ ]:
def english_ratio(text):
    letters = sum(c.isalpha() for c in text)
    return letters / max(len(text), 1)

def passes_final_ratio(text):
    return english_ratio(text) >= 0.6 and len(text) >= 10

after_l3 = [t for t in after_l2 if passes_final_ratio(t)]
print(f'Layer 3 -- kept {len(after_l3):,} / {len(after_l2):,}  ({len(after_l3)/len(after_l2)*100:.1f}%)')

## 6. Layer 4 - Minimum Word Count (NEW - BookCorpus-specific)

BookCorpus is sentence-split, so many rows are 1-4 word fragments like 'Yes.', 'He nodded.', 'Fine.'.
These carry almost no training signal for next-token prediction.

**Rule**: keep only rows with >= 5 words.

In [ ]:
# Inspect what would be dropped first
too_short = [t for t in after_l3 if len(t.split()) < 5]
print(f'Rows with < 5 words (to be dropped): {len(too_short):,}')
print('\nSamples of dropped rows:')
for s in random.sample(too_short, min(15, len(too_short))):
    print(f'  {repr(s)}')

In [ ]:
MIN_WORDS = 5

def passes_min_words(text, min_words=MIN_WORDS):
    return len(text.split()) >= min_words

after_l4 = [t for t in after_l3 if passes_min_words(t)]
print(f'Layer 4 -- kept {len(after_l4):,} / {len(after_l3):,}  ({len(after_l4)/len(after_l3)*100:.1f}%)')

## 7. Layer 5 - Boilerplate Pattern Filter (NEW - BookCorpus-specific)

Book structural markers that slipped through sentence splitting:
chapter headings, part labels, page numbers, ISBN strings, copyright lines, dedication pages.

In [ ]:
BOILERPLATE_PATTERNS = [
    r'^chapter\s+(\d+|one|two|three|four|five|six|seven|eight|nine|ten|\w+)\s*$',
    r'^part\s+(\d+|i{1,4}|one|two|three|four)\s*$',
    r'^\d+\s*$',
    r'isbn[\s:\-]*(\d[\d\-]{8,})',
    r'all rights reserved',
    r'copyright\s*(c)',
    r'printed in the',
    r'^(prologue|epilogue|acknowledgements?|dedication|preface|foreword|afterword)\s*$',
    r'^\s*[\*\-]{3,}\s*$',  # standalone divider lines only (*** alone on a line)
    r'^[\-]{3,}\s*$',
]

_boilerplate_re = re.compile('|'.join(BOILERPLATE_PATTERNS), re.IGNORECASE)

def is_boilerplate(text):
    return bool(_boilerplate_re.search(text.strip()))

boilerplate_caught = [t for t in after_l4 if is_boilerplate(t)]
print(f'Boilerplate rows caught: {len(boilerplate_caught):,}')
for s in boilerplate_caught[:20]:
    print(f'  {repr(s)}')

In [ ]:
after_l5 = [t for t in after_l4 if not is_boilerplate(t)]
print(f'Layer 5 -- kept {len(after_l5):,} / {len(after_l4):,}  ({len(after_l5)/len(after_l4)*100:.1f}%)')

### Layer 5b - Strip Scene-Break Markers

Lines like `*** she opened the door` are valid story text — the `***` is just a scene-break
convention in fiction. We keep the text but strip the leading asterisks.

In [ ]:
def strip_scene_markers(text):
    """Remove leading *** scene-break markers, keep the rest of the text."""
    cleaned = re.sub(r'^[\*]+\s*', '', text).strip()
    return cleaned if cleaned else text  # fall back to original if strip empties it

# Preview: show what changes
examples = [t for t in after_l5 if t.startswith('***')][:10]
for t in examples:
    print(f'  BEFORE: {repr(t)}')
    print(f'  AFTER : {repr(strip_scene_markers(t))}')
    print()

after_l5 = [strip_scene_markers(t) for t in after_l5]
print(f'Scene markers stripped. Total rows unchanged: {len(after_l5):,}')

## 8. Layer 6 - High-Punctuation / Dialogue Fragment Filter (NEW - BookCorpus-specific)

Ellipsis-heavy lines and bare dialogue fragments carry very little training signal.
We catch them via a high punctuation ratio.

**Inspect the samples below before locking in `MAX_PUNCT_RATIO`.**

In [ ]:
def punctuation_ratio(text):
    puncts = sum(not c.isalnum() and not c.isspace() for c in text)
    return puncts / max(len(text), 1)

MAX_PUNCT_RATIO = 0.3

high_punct = [(t, punctuation_ratio(t)) for t in after_l5 if punctuation_ratio(t) > MAX_PUNCT_RATIO]
high_punct.sort(key=lambda x: -x[1])

print(f'Rows with punctuation ratio > {MAX_PUNCT_RATIO}: {len(high_punct):,}')
print('\nHighest-ratio samples (adjust MAX_PUNCT_RATIO if these look like good text):')
for text, ratio in high_punct[:20]:
    print(f'  [{ratio:.2f}] {repr(text)}')

In [ ]:
def passes_punct_check(text, max_ratio=MAX_PUNCT_RATIO):
    return punctuation_ratio(text) <= max_ratio

after_l6 = [t for t in after_l5 if passes_punct_check(t)]
print(f'Layer 6 -- kept {len(after_l6):,} / {len(after_l5):,}  ({len(after_l6)/len(after_l5)*100:.1f}%)')

## 9. Layer 7 - Exact Deduplication (NEW - BookCorpus-specific)

`kd13/bookcorpus-clean` found ~40% of `rojagtap/bookcorpus` is duplicate content (same books re-uploaded).
We apply exact-match SHA-1 dedup here. At full scale, add MinHash near-duplicate removal too.

In [ ]:
def exact_dedup(texts):
    seen = set()
    unique = []
    for t in texts:
        key = hashlib.sha1(' '.join(t.lower().split()).encode()).hexdigest()
        if key not in seen:
            seen.add(key)
            unique.append(t)
    return unique

after_l7 = exact_dedup(after_l6)
print(f'Layer 7 -- kept {len(after_l7):,} / {len(after_l6):,}  ({len(after_l7)/len(after_l6)*100:.1f}%)')
print(f'Duplicates removed: {len(after_l6) - len(after_l7):,}')

## 10. Full Pipeline Summary

In [ ]:
stages = [
    ('Raw',                             len(texts_raw)),
    ('L1 - Sanity (alpha + word len)',  len(after_l1)),
    ('L2 - Language (English)',          len(after_l2)),
    ('L3 - Ratio + min char len',        len(after_l3)),
    ('L4 - Min word count (>=5)',        len(after_l4)),
    ('L5 - Boilerplate removal',         len(after_l5)),
    ('L6 - High punct filter',           len(after_l6)),
    ('L7 - Exact deduplication',         len(after_l7)),
]

print(f'{"Stage":<38} {"Count":>8}  {"Kept %":>7}  {"Dropped this step":>18}')
print('-' * 78)
for i, (name, count) in enumerate(stages):
    prev  = stages[i-1][1] if i > 0 else count
    kept  = count / stages[0][1] * 100
    drop  = (prev - count) / max(prev, 1) * 100 if i > 0 else 0
    print(f'{name:<38} {count:>8,}  {kept:>6.1f}%  {drop:>17.1f}%')

print()
print(f'Overall retention: {len(after_l7)/len(texts_raw)*100:.1f}% of raw data')

In [ ]:
names  = [s[0] for s in stages]
counts = [s[1] for s in stages]

plt.figure(figsize=(10, 5))
plt.barh(names[::-1], counts[::-1], color='steelblue')
plt.xlabel('Rows remaining')
plt.title('Cleaning funnel -- rojagtap/bookcorpus')
plt.tight_layout()
plt.show()

## 11. Quality Check - Sample the Final Dataset

In [ ]:
print('=== 20 random rows from the cleaned dataset ===')
for t in random.sample(after_l7, min(20, len(after_l7))):
    print(f'  {repr(t)}')

## 12. Decisions & Next Steps

After reviewing outputs above, record filtering decisions:

| Parameter | Value | Reason |
|-----------|-------|--------|
| `MIN_WORDS` | 5 | Removes 1-4 word fragments with no training signal |
| `MAX_PUNCT_RATIO` | 0.3 | Adjust up/down based on Section 8 output |
| Boilerplate patterns | See Layer 5 | Add/remove regex patterns based on Section 7 output |
| Dedup strategy | Exact SHA-1 + MinHash at full scale | ~40% of bookcorpus is duplicate books |

### Next step
Once decisions are finalised, update `book_corpus.py` to apply these filters when loading
`rojagtap/bookcorpus` -- or switch to `kd13/bookcorpus-clean` which already applied most of these
at the full 74M-row scale.